In [62]:
import pandas as pd
import sqlite3

In [63]:
df_raw = pd.read_csv('raw_ecommerce_data.csv')

In [64]:
df_raw.info()
df_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Order_ID       185 non-null    object
 1   Customer_Name  183 non-null    object
 2   Email          184 non-null    object
 3   Product        185 non-null    object
 4   Category       184 non-null    object
 5   Order_Date     185 non-null    object
 6   Quantity       185 non-null    int64 
 7   Unit_Price     185 non-null    object
 8   Amount         142 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB


,Order_ID,Customer_Name,Email,Product,Category,Order_Date,Quantity,Unit_Price,Amount
0,ORD-0036,Emma Brown,emma.brown@email.com,Monitor 24 inch,Electronics,24/04/2026,1,4131.00,"4,131.00"
1,ORD-0076,linda park,linda.park@email.com,Mechanical Keyboard,Electronics,"Apr 03, 2026",5,1669.50,8347.50
2,ORD-0101,john doe,john@email.com,Wireless Mouse,Electronics,11/04/2026,3,405.00,NaN
3,ORD-0059,Jane Smith,jane@email.com,Wireless Mouse,ELECTRONICS,13/03/2026,2,405.00,฿810.00
4,ORD-0038,PETER KIM,peter.kim@email.com,Gel Pen Set,Stationery,2026-02-28,5,85.50,427.50


In [65]:
dim_customer = df_raw[['Customer_Name','Email']].drop_duplicates()
df_raw.isnull().sum()

,0
Order_ID,0
Customer_Name,2
Email,1
Product,0
Category,1
Order_Date,0
Quantity,0
Unit_Price,0
Amount,43


In [66]:
dim_customer = dim_customer.reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1
dim_customer = dim_customer[['customer_id','Customer_Name','Email']]

In [67]:
fact_sales = pd.merge(df_raw,dim_customer,on=['Customer_Name','Email'],how='left')
fact_sales = fact_sales.drop(columns=['Customer_Name','Email'])

In [68]:
dim_product = df_raw[['Product', 'Category']].drop_duplicates()
dim_product = dim_product.reset_index(drop=True)
dim_product['product_id'] = dim_product.index + 1
dim_product = dim_product[['product_id', 'Product', 'Category']]

In [69]:
fact_sales = pd.merge(fact_sales, dim_product, on=['Product', 'Category'],how='left')
fact_sales = fact_sales.drop(columns=['Product', 'Category'])

In [70]:
dim_time = df_raw[['Order_Date']].drop_duplicates()
dim_time = dim_time.reset_index(drop=True)
dim_time['time_id'] = dim_time.index + 1
dim_time = dim_time[['time_id', 'Order_Date']]

In [71]:
fact_sales = pd.merge(fact_sales, dim_time,on=['Order_Date'], how='left')
fact_sales = fact_sales.drop(columns=['Order_Date'])

In [72]:
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

In [74]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_id INTEGER PRIMARY KEY,
        customer_name TEXT,
        email TEXT
    )
''')
conn.commit()

In [76]:
cursor.execute('PRAGMA foreign_keys = ON;')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_sales (
        transaction_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        product_id INTEGER,
        time_id INTEGER,
        quantity INTEGER,
        price REAL,
        FOREIGN KEY (customer_id) REFERENCES dim_customer (customer_id),
        FOREIGN KEY (product_id) REFERENCES dim_product (product_id)
    )
''')
conn.commit()

In [77]:
dim_customer.to_sql('dim_customer',con=conn, if_exists='replace', index=False)
fact_sales.to_sql('fact_sales',con=conn, if_exists='replace', index=False)
print('ETL Pipeline ran successfully')

ETL Pipeline ran successfully


In [82]:
cursor.execute('''
Select c.Customer_Name,Sum(f.amount) as Total_Spend
From fact_sales f
Join dim_customer c on f.customer_id = c.customer_id
Group by c.Customer_Name
Order by Total_Spend DESC
Limit 3
''')

result = cursor.fetchall()

column_names = [description[0] for description in cursor.description]

df_result = pd.DataFrame(result, columns=column_names)

print(df_result)

  Customer_Name  Total_Spend
0     Narin Dee      37597.5
1    Jane Smith      26985.0
2    Alice Wong      25009.5
